# Faster R-CNN (R50-FPN) - Yer seviyesi - catlak / yapisal hasar tespiti

**Katman:** Zorunlu

M3'te en yaygin tespit ailesi - literaturle karsilastirilabilirlik.

Bizim yontem: 2x YOLOv8 (catlak.pt + bina.pt) + hibrit kip, guven esigi 0,20, gecikme 153-174 ms.  
Butun modeller `common.py` uzerinden AYNI Roboflow seti / AYNI bolme / AYNI metrik.  
Metrik: mAP@50, mAP@50-95, Kesinlik, Duyarlilik, F1, sinif bazli AP + FPS/Params/GFLOPs.

In [ ]:
# === COLAB KURULUM ===  (yerelde calistirinca atlanir)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/QuakeMind'):
    !git clone --depth 1 https://github.com/UtBird/QuakeMind.git /content/QuakeMind
if IN_COLAB:
    %cd /content/QuakeMind/model_benchmarks/03_yer_seviyesi
    %pip install -q torch torchvision roboflow pycocotools
print('cwd:', os.getcwd())

## 1. Veri (Roboflow COCO export)

In [ ]:
import sys; from pathlib import Path; sys.path.insert(0, str(Path.cwd()))
import common as C, torch
C.CONFIG.update(dict(roboflow_api_key='BURAYA', roboflow_workspace='w', roboflow_project='p', roboflow_version=1))
coco_dir = C.download_dataset('coco')
dl = C.coco_loaders(coco_dir, C.CONFIG['img_size'], batch_size=4)
n_classes = len(__import__('json').load(open(coco_dir / 'train' / '_annotations.coco.json'))['categories']) + 1
print('sinif+bg:', n_classes)

## 2. Model

In [ ]:
MODEL_NAME = "faster_rcnn_r50fpn"
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
model = fasterrcnn_resnet50_fpn_v2(weights='DEFAULT')
model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features, n_classes)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; model.to(dev)

## 3. Egitim dongusu

In [ ]:
opt = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=5e-3, momentum=0.9, weight_decay=5e-4)
EPOCHS = 30
for ep in range(EPOCHS):
    model.train(); tot = 0
    for imgs, targets in dl['train']:
        imgs = [i.to(dev) for i in imgs]
        tg = [{'boxes': torch.tensor([o['bbox'] for o in t], dtype=torch.float32).reshape(-1,4).to(dev),
               'labels': torch.tensor([o['category_id'] for o in t], dtype=torch.int64).to(dev)} for t in targets]
        # NOT: Roboflow COCO bbox [x,y,w,h] -> [x1,y1,x2,y2] donusumu gerekir
        for d in tg:
            if len(d['boxes']): d['boxes'][:, 2:] += d['boxes'][:, :2]
        loss = sum(model(imgs, tg).values())
        opt.zero_grad(); loss.backward(); opt.step(); tot += float(loss)
    print(f'epoch {ep+1} loss {tot/len(dl["train"]):.3f}')

## 4. Degerlendirme (torchmetrics mAP) + kaydet

In [ ]:
# pip install -q torchmetrics
from torchmetrics.detection import MeanAveragePrecision
metric = MeanAveragePrecision(box_format='xyxy')
model.eval()
with torch.no_grad():
    for imgs, targets in dl.get('test') or dl['valid']:
        imgs = [i.to(dev) for i in imgs]
        preds = model(imgs)
        gts = [{'boxes': torch.tensor([o['bbox'] for o in t]).reshape(-1,4),
                'labels': torch.tensor([o['category_id'] for o in t])} for t in targets]
        for g in gts:
            if len(g['boxes']): g['boxes'][:, 2:] += g['boxes'][:, :2]
        metric.update([{k: v.cpu() for k, v in p.items()} for p in preds], gts)
r = metric.compute()
metrics = dict(map50=float(r['map_50']), map50_95=float(r['map']),
               precision=float('nan'), recall=float(r['mar_100']), f1=float('nan'))
eff = dict(params_m=sum(p.numel() for p in model.parameters())/1e6, gflops=float('nan'),
           infer_ms=float('nan'), fps=float('nan'))
C.save_result(MODEL_NAME, metrics, eff)
print(metrics)